In [4]:
import os
import pandas as pd
import glob
import datetime

def filter_csv_files_by_date(
    input_folder,
    date_filter=None,
    after_midday=False,
    pattern="*.csv"
):
    """
    Filters CSV files in a folder by creation date
    
    Parameters:
    -----------
    input_folder : str
        Path to folder containing CSV files
    date_filter : datetime.date
        Filter files created on this date (default: today)
    after_midday : bool
        If True, only include files created after noon
    pattern : str
        File pattern to match (default: "*.csv")
        
    Returns:
    --------
    list: Filtered file paths
    """
    # Use today's date if not specified
    if date_filter is None:
        date_filter = datetime.date.today()
    
    print(f"Looking for CSV files in: {input_folder}")
    
    # Get all matching files in the folder
    all_files = glob.glob(os.path.join(input_folder, pattern))
    print(f"Found {len(all_files)} total files matching pattern '{pattern}'")
    
    # Create time threshold
    if after_midday:
        time_threshold = datetime.datetime.combine(date_filter, datetime.time(12, 0))
        time_desc = "after midday"
    else:
        time_threshold = datetime.datetime.combine(date_filter, datetime.time(0, 0))
        time_desc = "all day"
    
    # Filter files by creation time
    filtered_files = []
    for file_path in all_files:
        # Get creation timestamp
        file_creation_time = datetime.datetime.fromtimestamp(os.path.getctime(file_path))
        
        # Apply filter
        if file_creation_time.date() == date_filter and file_creation_time >= time_threshold:
            filtered_files.append(file_path)
    
    print(f"Found {len(filtered_files)} files created {time_desc} on {date_filter}")
    
    # Sort files to maintain order
    filtered_files.sort()
    
    return filtered_files


def combine_csv_files(
    file_list,
    output_folder,
    target_chunk_size=10000000,
    prefix="region"
):
    """
    Combines CSV files into larger chunks of specified size.
    
    Parameters:
    -----------
    file_list : list
        List of CSV file paths to combine
    output_folder : str
        Path to save combined CSV files
    target_chunk_size : int
        Number of rows per output file
    prefix : str
        Prefix for output CSV files (e.g., "africa", "europe")
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    if not file_list:
        print("No files to process")
        return
    
    print(f"Will process {len(file_list)} files")
    
    # Initialize variables for processing
    current_batch = []
    current_row_count = 0
    batch_number = 1
    start_row = 0
    
    # Process each file
    for file_path in file_list:
        print(f"Reading: {os.path.basename(file_path)}")
        
        # Read and count rows without loading entire file
        with open(file_path, 'r') as f:
            # Count lines but subtract 1 for header
            file_rows = sum(1 for _ in f) - 1
            
        # Check if adding this file would exceed target chunk size
        if current_row_count + file_rows > target_chunk_size and current_batch:
            # Combine and save current batch
            end_row = start_row + current_row_count
            output_file = os.path.join(output_folder, f"{prefix}_{start_row}_to_{end_row}.csv")
            
            print(f"Saving batch {batch_number} with {current_row_count} rows as: {os.path.basename(output_file)}")
            
            # Combine files in batch
            combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
            combined_df.to_csv(output_file, index=False)
            
            # Reset for next batch
            current_batch = []
            start_row = end_row
            current_row_count = 0
            batch_number += 1
        
        # Add file to current batch
        current_batch.append(file_path)
        current_row_count += file_rows
    
    # Process any remaining files in the last batch
    if current_batch:
        end_row = start_row + current_row_count
        output_file = os.path.join(output_folder, f"{prefix}_{start_row}_to_{end_row}.csv")
        
        print(f"Saving final batch with {current_row_count} rows as: {os.path.basename(output_file)}")
        
        # Combine files in batch
        combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
        combined_df.to_csv(output_file, index=False)
    
    print(f"Processing complete! Created {batch_number} combined CSV files")



In [5]:
# input_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_geo_csvs"
# output_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs"
    
# # First filter files
# csv_files = filter_csv_files_by_date(
#     input_folder=input_folder,
#     date_filter= datetime.date(2025, 6, 4),
#     after_midday=False, 
#     pattern="*railways*" # Get files created after noon today
#     )
# print(csv_files)

In [6]:

# Example usage
if __name__ == "__main__":

    input_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_geo_csvs"
    output_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs"
    
    # patterns = [
    #     "*_waterways.gpkg*",
    #     "*_ferry_routes.gpkg*",
    #     "*_boat_access.gpkg*",
    #     "*_railways.gpkg*",
    #     # "*_highways.gpkg*"
    # ]
    patterns = [
        # "*_waterways*",
        "*_ferry_routes*",
        "*_boat_access*",
        "*_railways*",
        # "*_highways.gpkg*"
    ]

    for pattern in patterns:
            # First filter files
        csv_files = filter_csv_files_by_date(
            input_folder=input_folder,
            date_filter= datetime.date(2025, 6, 6),
            after_midday=False, 
            pattern=pattern # Get files created after noon today
            )
        print(csv_files)    
    
        dataset_prefix = "global"+"_"+pattern.split("_")[1].replace("*", "").replace(".gpkg*", "")
        
        print(f"Processing files with output prefix: {dataset_prefix}")

        # Then combine them
        combine_csv_files(
            file_list=csv_files,
            output_folder=output_folder,
            target_chunk_size=10000000,
            prefix=dataset_prefix
        )
    print("All files processed successfully!")

Looking for CSV files in: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_geo_csvs
Found 8 total files matching pattern '*_ferry_routes*'
Found 8 files created all day on 2025-06-06
['C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_geo_csvs\\africa_ferry_routes_0_to_2617_of_2617.csv', 'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_geo_csvs\\antarctica_ferry_routes_0_to_3_of_3.csv', 'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_geo_csvs\\asia_ferry_routes_0_to_13648_of_13648.csv', 'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\

C:\Users\Arnell\AppData\Local\Temp\ipykernel_15548\2131679694.py:140: DtypeWarning: Columns (6,10,33,40,41,43) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
C:\Users\Arnell\AppData\Local\Temp\ipykernel_15548\2131679694.py:140: DtypeWarning: Columns (10,12,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
C:\Users\Arnell\AppData\Local\Temp\ipykernel_15548\2131679694.py:140: DtypeWarning: Columns (6,12,13,14,15,30,31,33,34,41,42,44,46) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
C:\Users\Arnell\AppData\Local\Temp\ipykernel_15548\2131679694.py:140: DtypeWarning: Columns (4,6,10,11,20,29,33,37,40,41,43) have mixed types. Specify dtype option on impo

KeyboardInterrupt: 

Dtype error - fix 

In [7]:
def combine_csv_files(
    file_list,
    output_folder,
    target_chunk_size=10000000,
    prefix="region"
):
    """Combines CSV files into larger chunks of specified size."""
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    if not file_list:
        print("No files to process")
        return
    
    print(f"Will process {len(file_list)} files")
    
    # Initialize variables for processing
    current_batch = []
    current_row_count = 0
    batch_number = 1
    start_row = 0
    
    # Process each file
    for file_path in file_list:
        print(f"Reading: {os.path.basename(file_path)}")
        
        # Read and count rows without loading entire file
        with open(file_path, 'r') as f:
            # Count lines but subtract 1 for header
            file_rows = sum(1 for _ in f) - 1
            
        # Check if adding this file would exceed target chunk size
        if current_row_count + file_rows > target_chunk_size and current_batch:
            # Combine and save current batch
            end_row = start_row + current_row_count
            output_file = os.path.join(output_folder, f"{prefix}_{start_row}_to_{end_row}.csv")
            
            print(f"Saving batch {batch_number} with {current_row_count} rows as: {os.path.basename(output_file)}")
            
            # IMPROVED: Use csv module for more reliable geometry handling
            import csv
            
            # First read headers from first file
            with open(current_batch[0], 'r') as f:
                headers = next(csv.reader(f))
            
            # Open output file
            with open(output_file, 'w', newline='') as outfile:
                writer = csv.writer(outfile, quoting=csv.QUOTE_NONNUMERIC)
                writer.writerow(headers)
                
                # Process each input file
                for input_file in current_batch:
                    with open(input_file, 'r') as infile:
                        reader = csv.reader(infile)
                        next(reader)  # Skip header
                        
                        # Copy rows with proper quoting
                        for row in reader:
                            writer.writerow(row)
            
            # Reset for next batch
            current_batch = []
            start_row = end_row
            current_row_count = 0
            batch_number += 1
        
        # Add file to current batch
        current_batch.append(file_path)
        current_row_count += file_rows
    
    # Process any remaining files in the last batch
    if current_batch:
        end_row = start_row + current_row_count
        output_file = os.path.join(output_folder, f"{prefix}_{start_row}_to_{end_row}.csv")
        
        print(f"Saving final batch with {current_row_count} rows as: {os.path.basename(output_file)}")
        
        # IMPROVED: Use same csv module approach as above
        import csv
            
        # First read headers from first file
        with open(current_batch[0], 'r') as f:
            headers = next(csv.reader(f))
        
        # Open output file
        with open(output_file, 'w', newline='') as outfile:
            writer = csv.writer(outfile, quoting=csv.QUOTE_NONNUMERIC)
            writer.writerow(headers)
            
            # Process each input file
            for input_file in current_batch:
                with open(input_file, 'r') as infile:
                    reader = csv.reader(infile)
                    next(reader)  # Skip header
                    
                    # Copy rows with proper quoting
                    for row in reader:
                        writer.writerow(row)
    
    print(f"Processing complete! Created {batch_number} combined CSV files")

In [8]:

# Example usage
if __name__ == "__main__":

    input_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_geo_csvs"
    output_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs"
    
    # patterns = [
    #     "*_waterways.gpkg*",
    #     "*_ferry_routes.gpkg*",
    #     "*_boat_access.gpkg*",
    #     "*_railways.gpkg*",
    #     # "*_highways.gpkg*"
    # ]
    patterns = [
        # "*_waterways*",
        "*_ferry_routes*",
        "*_boat_access*",
        "*_railways*",
        # "*_highways.gpkg*"
    ]

    for pattern in patterns:
            # First filter files
        csv_files = filter_csv_files_by_date(
            input_folder=input_folder,
            date_filter= datetime.date(2025, 6, 6),
            after_midday=False, 
            pattern=pattern # Get files created after noon today
            )
        print(csv_files)    
    
        dataset_prefix = "global"+"_"+pattern.split("_")[1].replace("*", "").replace(".gpkg*", "")
        
        print(f"Processing files with output prefix: {dataset_prefix}")

        # Then combine them
        combine_csv_files(
            file_list=csv_files,
            output_folder=output_folder,
            target_chunk_size=10000000,
            prefix=dataset_prefix
        )
    print("All files processed successfully!")

Looking for CSV files in: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_geo_csvs
Found 8 total files matching pattern '*_ferry_routes*'
Found 8 files created all day on 2025-06-06
['C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_geo_csvs\\africa_ferry_routes_0_to_2617_of_2617.csv', 'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_geo_csvs\\antarctica_ferry_routes_0_to_3_of_3.csv', 'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_geo_csvs\\asia_ferry_routes_0_to_13648_of_13648.csv', 'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\

Quick version w hard coded filter (temp)

In [ ]:
# def combine_todays_csvs(
#     input_folder,
#     output_folder,
#     target_chunk_size=10000000,
#     prefix="region",
#     after_midday=True
# ):
#     """
#     Combines CSV files created today (optionally after midday) into larger chunks.
    
#     Parameters:
#     -----------
#     input_folder : str
#         Path to folder containing CSV files
#     output_folder : str
#         Path to save combined CSV files
#     target_chunk_size : int
#         Number of rows per output file
#     prefix : str
#         Prefix for output CSV files (e.g., "africa", "europe")
#     after_midday : bool
#         If True, only process files created after noon today
#     """
#     import os
#     import pandas as pd
#     import glob
#     import datetime
    
#     # Create output folder if it doesn't exist
#     os.makedirs(output_folder, exist_ok=True)
    
#     # Create datetime filter for today
#     today = datetime.date.today()
    
#     # Create midday threshold if needed
#     if after_midday:
#         midday_threshold = datetime.datetime.combine(today, datetime.time(12, 0))
#         time_desc = "after midday"
#     else:
#         midday_threshold = datetime.datetime.combine(today, datetime.time(0, 0))
#         time_desc = "all day"
    
#     print(f"Looking for CSV files in: {input_folder}")
#     print(f"Time filter: {today} {time_desc}")
    
#     # Get all CSV files in the folder
#     all_csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
#     print(f"Found {len(all_csv_files)} total CSV files")
    
#     # Filter files created today after midday
#     filtered_files = []
#     for file_path in all_csv_files:
#         # Get full datetime of creation
#         file_creation_time = datetime.datetime.fromtimestamp(os.path.getctime(file_path))
        
#         # Filter based on threshold
#         if file_creation_time >= midday_threshold:
#             filtered_files.append(file_path)
    
#     print(f"Found {len(filtered_files)} CSV files created {time_desc} on {today}")
    
#     if not filtered_files:
#         print("No files to process")
#         return
    
#     # Sort files (to maintain order)
#     filtered_files.sort()
    
#     # Initialize variables for processing
#     current_batch = []
#     current_row_count = 0
#     batch_number = 1
#     start_row = 0
    
#     # Process each file
#     for file_path in filtered_files:
#         print(f"Reading: {os.path.basename(file_path)}")
        
#         # Read and count rows without loading entire file
#         with open(file_path, 'r') as f:
#             # Count lines but subtract 1 for header
#             file_rows = sum(1 for _ in f) - 1
            
#         # Check if adding this file would exceed target chunk size
#         if current_row_count + file_rows > target_chunk_size and current_batch:
#             # Combine and save current batch
#             end_row = start_row + current_row_count
#             output_file = os.path.join(output_folder, f"{prefix}_{start_row}_to_{end_row}.csv")
            
#             print(f"Saving batch {batch_number} with {current_row_count} rows as: {os.path.basename(output_file)}")
            
#             # Combine files in batch
#             combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
#             combined_df.to_csv(output_file, index=False)
            
#             # Reset for next batch
#             current_batch = []
#             start_row = end_row
#             current_row_count = 0
#             batch_number += 1
        
#         # Add file to current batch
#         current_batch.append(file_path)
#         current_row_count += file_rows
    
#     # Process any remaining files in the last batch
#     if current_batch:
#         end_row = start_row + current_row_count
#         output_file = os.path.join(output_folder, f"{prefix}_{start_row}_to_{end_row}.csv")
        
#         print(f"Saving final batch with {current_row_count} rows as: {os.path.basename(output_file)}")
        
#         # Combine files in batch
#         combined_df = pd.concat([pd.read_csv(f) for f in current_batch], ignore_index=True)
#         combined_df.to_csv(output_file, index=False)
    
#     print(f"Processing complete! Created {batch_number} combined CSV files")

# # Example usage
# if __name__ == "__main__":
#     input_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\roads\osm\osm_regional_250521\geo_csvs"
#     output_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\roads\osm\osm_regional_250521\combined_csvs"
    
#     combine_todays_csvs(
#         input_folder=input_folder,
#         output_folder=output_folder,
#         target_chunk_size=10000000,
#         prefix="europe",
#         after_midday=True  # Only process files created after noon today
#     )

Looking for CSV files in: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\roads\osm\osm_regional_250521\geo_csvs
Time filter: 2025-05-29 after midday
Found 73 total CSV files
Found 52 CSV files created after midday on 2025-05-29
Reading: albania_0_to_169935_of_169935.csv
Reading: andorra_0_to_8038_of_8038.csv
Reading: austria_0_to_2401937_of_2401937.csv
Reading: azores_0_to_37964_of_37964.csv
Reading: belarus_0_to_944214_of_944214.csv
Reading: belgium_0_to_1407418_of_1407418.csv
Reading: bosnia_herzegovina_0_to_271015_of_271015.csv
Reading: bulgaria_0_to_575524_of_575524.csv
Reading: croatia_0_to_552416_of_552416.csv
Reading: cyprus_0_to_180472_of_180472.csv
Reading: czech_republic_0_to_1779557_of_1779557.csv
Reading: denmark_0_to_1282957_of_1282957.csv
Reading: estonia_0_to_310641_of_310641.csv
Reading: faroe_islands_0_to_23688_of_23688.csv
Reading: finland_0_to_2029062_of_2029062.csv
Saving batch 1 with 9945776 r